In [140]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
import faiss

In [141]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [142]:
docs = []

for i in range(10):
    with open(f"docs/text{i}.txt", "r") as f:
        payload = f.read()
    docs.append(payload.strip())
print(f"число документов: {len(docs)}")

print("примеры")
for i in range(4):
    print("пример", i, ":", docs[i][:100])


"""
База знаний про основы ML, подходит для retrieval, т.к. содержит определения и объяснения.
"""

число документов: 10
примеры
пример 0 : Machine learning (ML) is a field of study in artificial intelligence concerned with the development 
пример 1 : Dimensionality reduction, or dimension reduction, is the transformation of data from a high-dimensio
пример 2 : In machine learning, a common task is the study and construction of algorithms that can learn from a
пример 3 : Artificial intelligence (AI) is the capability of computational systems to perform tasks typically a


'\nБаза знаний про основы ML, подходит для retrieval, т.к. содержит определения и объяснения.\n'

База знаний про основы ML, подходит для retrieval, т.к. содержит определения и объяснения.

In [143]:
def chunk_text(text, chunk_size=100, overlap=20):
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start+chunk_size])
        start += chunk_size - overlap
    return chunks

In [144]:
all_chunks = []
chunk_to_doc = []

for i, doc in enumerate(docs):
    chunks = chunk_text(doc)
    all_chunks.extend(chunks)
    chunk_to_doc.extend([i]*len(chunks))

print("примеры разбиения на чанки")
print(all_chunks[:5])
print(len(all_chunks))

примеры разбиения на чанки
['Machine learning (ML) is a field of study in artificial intelligence concerned with the development ', 'ith the development and study of statistical algorithms that can learn from data and generalize to u', ' and generalize to unseen data, and thus perform tasks without explicit programming language instruc', 'ing language instructions.[1] Within a subdiscipline of machine learning, advances in the field of d', 'es in the field of deep learning have allowed neural networks, a class of statistical algorithms, to']
700


Чанкинг выполняется разбиением каждого документа на последовательные фрагменты фиксированной длины 100 символов с перекрытием 20 символов между соседними частями. Размер чанка выбран так, чтобы сохранять локальный смысл текста, а overlap используется для сохранения контекста на границах разбиения и уменьшения потерь информации при разрезании предложений.

In [145]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(all_chunks).toarray()

In [146]:
dim = X.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(X.astype(np.float32))

In [147]:
def search(query, k=3):
    q_vec = vectorizer.transform([query]).toarray().astype(np.float32)
    D, I = index.search(q_vec, k)
    return I[0]

In [148]:
queries = ["machine learning", "deep learning", "large language model"]

for q in queries:
    idxs = search(q)
    print("запрос", q)
    for i in idxs:
        print(all_chunks[i])

запрос machine learning
[6]
a compression В§ Machine learning.[edit]
There is a close connection between machine learning and co
uses many machine learning methods, but with different goals; on the other hand, machine learning al
запрос deep learning
hallow learning from deep learning, but most researchers agree that deep learning involves CAP depth
[6]
se in deep generative models such as the nodes in deep belief networks and deep Boltzmann machines.[
запрос large language model
A large language model (LLM) is a computational model designed to perform natural language processin
[6]
f large models are language models or multimodal models with language capacity.
Before the emergence


In [149]:
eval_data = [
    {"query": "what is Machine learning", "expected_doc": 0},
    {"query": "what is Supervised learning", "expected_doc": 0},
    {"query": "what is Dimensionality reduction", "expected_doc": 1},
    {"query": "what is test and validation sets", "expected_doc": 2},
    {"query": "what is Artificial intelligence", "expected_doc": 3},
    {"query": "what is neural network", "expected_doc": 4},
    {"query": "what is deep learning", "expected_doc": 5},
    {"query": "what is Data mining", "expected_doc": 6},
    {"query": "what is artificial neuron", "expected_doc": 7},
]

In [150]:
def hit_at_k(retrieved, expected):
    return int(expected in retrieved)

def recall_at_k(retrieved, expected):
    return int(expected in retrieved)

In [167]:
results = []

k = 3

for item in eval_data:
    idxs = search(item["query"], 3)
    retrieved_docs = [chunk_to_doc[i] for i in idxs]
    
    hit = hit_at_k(retrieved_docs, item["expected_doc"])
    
    results.append({
        "query": item["query"],
        "expected_source": item["expected_doc"],
        "retrieved_sources": retrieved_docs,
        "hit_at_k": hit
    })

df_eval = pd.DataFrame(results)
df_eval.to_csv("artifacts/retrieval_eval.csv", index=False)
print("hit_at_k", df_eval["hit_at_k"].sum(), "/9")
print("recall@k:", df_eval["hit_at_k"].mean())
display(df_eval)

hit_at_k 9 /9
recall@k: 1.0


,query,expected_source,retrieved_sources,hit_at_k
0,what is Machine learning,0,"[1, 3, 0]",1
1,what is Supervised learning,0,"[1, 0, 0]",1
2,what is Dimensionality reduction,1,"[1, 1, 1]",1
3,what is test and validation sets,2,"[1, 2, 2]",1
4,what is Artificial intelligence,3,"[1, 3, 0]",1
5,what is neural network,4,"[1, 4, 4]",1
6,what is deep learning,5,"[1, 5, 3]",1
7,what is Data mining,6,"[1, 3, 6]",1
8,what is artificial neuron,7,"[1, 7, 7]",1


In [164]:
results_new = []

k = 1

for item in eval_data:
    idxs = search(item["query"], k)
    retrieved_docs = [chunk_to_doc[i] for i in idxs]
    
    hit = hit_at_k(retrieved_docs, item["expected_doc"])
    
    results_new.append({
        "query": item["query"],
        "expected_source": item["expected_doc"],
        "retrieved_sources": retrieved_docs,
        "hit_at_k": hit
    })

df_eval_new = pd.DataFrame(results_new)
display(df_eval_new)
print("кол-во попаданий после изменения top_k", df_eval_new["hit_at_k"].sum())
"""
При уменьшении top_k до значения 1 кол-во попаданий резко снизилось из-за того, что первая статья довольно большая и содержит часть ответа на запросы
"""

,query,expected_source,retrieved_sources,hit_at_k
0,what is Machine learning,0,[1],0
1,what is Supervised learning,0,[1],0
2,what is Dimensionality reduction,1,[1],1
3,what is test and validation sets,2,[1],0
4,what is Artificial intelligence,3,[1],0
5,what is neural network,4,[1],0
6,what is deep learning,5,[1],0
7,what is Data mining,6,[1],0
8,what is artificial neuron,7,[1],0


кол-во попаданий после изменения top_k 1


'\nПри уменьшении top_k до значения 1 кол-во попаданий резко снизилось из-за того, что первая статья довольно большая и содержит часть ответа на запросы\n'

При уменьшении top_k до значения 1 кол-во попаданий резко снизилось из-за того, что первая статья довольно большая и содержит часть ответа на запросы

In [153]:
for i in range(10, 13):
    with open(f"docs/text{i}.txt", "r") as f:
        payload = f.read()
    docs.append(payload.strip())
print("кол-во документов", len(docs))

all_chunks = []
chunk_to_doc = []

for i, doc in enumerate(docs):
    chunks = chunk_text(doc)
    all_chunks.extend(chunks)
    chunk_to_doc.extend([i]*len(chunks))

vectorizer2 = TfidfVectorizer()
X2 = vectorizer2.fit_transform(all_chunks).toarray()
dim2 = X2.shape[1]
index2 = faiss.IndexFlatL2(dim2)
index2.add(X2.astype(np.float32))

def search2(query, k=3):
    q_vec = vectorizer2.transform([query]).toarray().astype(np.float32)
    D, I = index2.search(q_vec, k)
    return I[0]

кол-во документов 13


In [168]:
results_after_adding_docs = []

k = 3

for item in eval_data:
    idxs = search2(item["query"], 3)
    retrieved_docs = [chunk_to_doc[i] for i in idxs]
    
    hit = hit_at_k(retrieved_docs, item["expected_doc"])
    
    results_after_adding_docs.append({
        "query": item["query"],
        "expected_source": item["expected_doc"],
        "retrieved_sources": retrieved_docs,
        "hit_at_k": hit
    })

df_eval_after_adding_docs = pd.DataFrame(results_after_adding_docs)
print("кол-во попаданий", df_eval_after_adding_docs["hit_at_k"].sum())

retrieval_before_after_update = {"query": [], "before_retrieved_sources": [], "after_retrieved_sources": [], "changed": []}
for i in range(len(df_eval_after_adding_docs)):
    df_eval_row = df_eval.iloc[i]
    df_eval_after_row = df_eval_after_adding_docs.iloc[i]

    retrieval_before_after_update["query"].append(df_eval_row["query"])
    retrieval_before_after_update["before_retrieved_sources"].append(df_eval_row["retrieved_sources"])
    retrieval_before_after_update["after_retrieved_sources"].append(df_eval_after_row["retrieved_sources"])
    retrieval_before_after_update["changed"].append(list(set(df_eval_after_row["retrieved_sources"]) ^ set(df_eval_row["retrieved_sources"])))

    if set(df_eval_row["retrieved_sources"]) != set(df_eval_after_row["retrieved_sources"]):
        print("запрос", df_eval_row["query"], "источники до обновления", df_eval_row["retrieved_sources"], "источники после", df_eval_after_row["retrieved_sources"])

df_retrieval_before_after_update = pd.DataFrame(retrieval_before_after_update)
df_retrieval_before_after_update.to_csv("artifacts/retrieval_before_after_update.csv", index=False)

кол-во попаданий 6
запрос what is Machine learning источники до обновления [1, 3, 0] источники после [11, 11, 10]
запрос what is Supervised learning источники до обновления [1, 0, 0] источники после [1, 11, 0]
запрос what is Dimensionality reduction источники до обновления [1, 1, 1] источники после [1, 1, 0]
запрос what is Artificial intelligence источники до обновления [1, 3, 0] источники после [1, 10, 3]
запрос what is deep learning источники до обновления [1, 5, 3] источники после [11, 11, 1]
запрос what is Data mining источники до обновления [1, 3, 6] источники после [1, 12, 12]


In [155]:
def mini_rag(query, k=3):
    idxs = search2(query, k)
    context = " ".join([all_chunks[i] for i in idxs])
    
    answer = context[:200]
    
    return answer, idxs

In [160]:
rag_results = []

for item in eval_data[:5]:
    q = item["query"]
    ans, idxs = mini_rag(q)
    retrieved_docs = [chunk_to_doc[i] for i in idxs]
    
    rag_results.append({
        "question": q,
        "answer": ans,
        "retrieved_sources": retrieved_docs
    })

pd.DataFrame(rag_results).to_csv("artifacts/rag_examples.csv", index=False)
print(rag_results[:5])

"""
В ряде примеров mini-RAG наблюдаются ошибки, связанные в основном с неточным retrieval и качеством чанкинга. Для запроса про Machine learning возвращаются фрагменты, смешивающие разные темы (например, deep learning), что связано с тем, что TF-IDF плохо разделяет близкие по смыслу понятия. В случае Supervised learning и Dimensionality reduction в ответах присутствуют нерелевантные или соседние по тематике чанки, что указывает на слабую точность поиска и пересечение тем в базе знаний. Для test and validation sets заметны обрывки текста и потеря контекста, что связано с простым символьным чанкингом. В целом основные проблемы вызваны ограниченной точностью TF-IDF и разбиением текста, которое иногда разрушает смысловые фрагменты.
"""

[{'question': 'what is Machine learning', 'answer': 'ore.\n\nWhat is the difference between machine learning and deep learning?\nMachine learning is a type  What is deep learning?\nDeep learning is a type of machine learning that can recognize complex patte', 'retrieved_sources': [11, 11, 10]}, {'question': 'what is Supervised learning', 'answer': '[6] ore.\n\nWhat is the difference between machine learning and deep learning?\nMachine learning is a type  .[44][45][46]\n\nSupervised learning\nMain article: Supervised learning\n\nA support-vector machine ', 'retrieved_sources': [1, 11, 0]}, {'question': 'what is Dimensionality reduction', 'answer': 'Dimensionality reduction, or dimension reduction, is the transformation of data from a high-dimensio [6] Central applications of unsupervised machine learning include clustering, dimensionality reduct', 'retrieved_sources': [1, 1, 0]}, {'question': 'what is test and validation sets', 'answer': '[6]  evaluated using вЂњnewвЂќ examples from the

'\nВ ряде примеров mini-RAG наблюдаются ошибки, связанные в основном с неточным retrieval и качеством чанкинга. Для запроса про Machine learning возвращаются фрагменты, смешивающие разные темы (например, deep learning), что связано с тем, что TF-IDF плохо разделяет близкие по смыслу понятия. В случае Supervised learning и Dimensionality reduction в ответах присутствуют нерелевантные или соседние по тематике чанки, что указывает на слабую точность поиска и пересечение тем в базе знаний. Для test and validation sets заметны обрывки текста и потеря контекста, что связано с простым символьным чанкингом. В целом основные проблемы вызваны ограниченной точностью TF-IDF и разбиением текста, которое иногда разрушает смысловые фрагменты.\n'

В ряде примеров mini-RAG наблюдаются ошибки, связанные в основном с неточным retrieval и качеством чанкинга. Для запроса про Machine learning возвращаются фрагменты, смешивающие разные темы (например, deep learning), что связано с тем, что TF-IDF плохо разделяет близкие по смыслу понятия. В случае Supervised learning и Dimensionality reduction в ответах присутствуют нерелевантные или соседние по тематике чанки, что указывает на слабую точность поиска и пересечение тем в базе знаний. Для test and validation sets заметны обрывки текста и потеря контекста, что связано с простым символьным чанкингом. В целом основные проблемы вызваны ограниченной точностью TF-IDF и разбиением текста, которое иногда разрушает смысловые фрагменты.